# Parking Lot Vehicle Monitoring with YOLO

**Goal of this notebook**

Build an end-to-end pipeline, built for Kaggle Notebooks, that uses the GPU by default (GPU `0` — the first of the T4 x2) and can be switched to CPU with a single boolean flag (Section 3), that:

1. Trains a YOLO object detector on a vehicle-image dataset (cars + motorcycles).
2. Uses the trained detector on periodic photos of a small parking lot to:
   - Detect whether **specific parking spots** (calibrated once, by hand) are occupied or empty.
   - Track vehicles across photos to determine when a car **enters**, **leaves**, and **how long it stayed**.
3. As a final, separate module: given a single parking-lot photo, outputs for each detected vehicle its **dominant color**, and for cars specifically, a best-guess **brand and model** (zero-shot, no extra training required).

**Design decisions locked in from our discussion (so future-you remembers why):**

- Training uses a **generic, already-YOLO-formatted vehicle dataset** (Roboflow's `Vehicles-OpenImages`, classes: `Car`, `Motorcycle`, `Bus`, `Truck`, `Ambulance`). It is not parking-lot-specific, so accuracy on your exact camera angle may need a later fine-tuning pass with your own labeled photos.
- Everything runs on the **GPU by default** (GPU `0` on Kaggle's T4 x2). One boolean flag (Section 3) switches the entire notebook to CPU — nothing else changes.
- Parking spots are defined as **polygons, not axis-aligned boxes**, because a wide/fisheye camera angle distorts spot shapes into trapezoids (see calibration section).
- The **walkway area is intentionally excluded** from the spot list — it is not a real parking spot, just floor space.
- The **motorcycle spot only checks for presence** of a motorcycle (no brand/model needed there) — color is a nice-to-have.
- **No OCR / timestamp reading is baked into training or detection.** The burned-in camera timestamp is never read or used to alter training images in any way. For the entry/exit/duration tracking demo, timestamps come from an external list you provide (e.g. derived from filenames) — fully decoupled from the model pipeline.

Let's get started.

## 1. Install dependencies

We need:
- `ultralytics` — YOLO training/inference (YOLOv8).
- `roboflow` — to download the training dataset in YOLO format.
- `shapely` — polygon geometry for parking-spot occupancy checks.
- `opencv-python-headless` — image loading/drawing.
- `scikit-learn` — K-Means for dominant-color extraction.
- `open_clip_torch` — pretrained CLIP model, used **zero-shot** (no training) for brand/model guessing in the final module.
- `pandas` — building the entry/exit/duration log as a table.

Colab already has `torch`, `numpy`, `matplotlib`, and `Pillow` preinstalled, so we don't reinstall those.

In [ ]:
!pip install -q ultralytics roboflow shapely opencv-python-headless scikit-learn open_clip_torch pandas

## 2. Imports

Standard imports for the whole notebook. Nothing here trains or downloads anything yet.

In [ ]:
import os
import json
import glob
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from shapely.geometry import Polygon, box as shapely_box

import torch
from ultralytics import YOLO

print("Setup OK. Libraries imported.")

## 3. Choose the device (GPU / CPU)

One boolean controls the device for the **entire** notebook — every training and inference cell below reads `FORCE_DEVICE` from this cell:

- **`USE_CPU = False` (default)** → uses the GPU, device `0` (the first GPU of Kaggle's T4 x2).
- **`USE_CPU = True`** → forces `"cpu"` everywhere.

Flip that one flag and nothing else in the notebook needs to change.

In [ ]:
# Single switch for the whole notebook:
#   False -> GPU (device 0 — the first GPU of Kaggle's T4 x2)
#   True  -> CPU
USE_CPU = False

if USE_CPU:
    FORCE_DEVICE = "cpu"
else:
    FORCE_DEVICE = 0  # first GPU (T4 x2 on Kaggle)

# Model selection — change this single variable to switch models
MODEL_NAME = "yolov8n.pt"  # e.g. "yolov8n.pt", "yolov8s.pt", "yolov11n.pt", etc.

print(f"FORCE_DEVICE set to: {FORCE_DEVICE!r}")
print(f"MODEL_NAME: {MODEL_NAME}")
if not USE_CPU and torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  - cuda:{i} -> {torch.cuda.get_device_name(i)}")

## 4. Get the training dataset

We use **`Vehicles-OpenImages`**, a public dataset on Roboflow Universe with the classes: `Ambulance`, `Bus`, `Car`, `Motorcycle`, `Truck`. This gives us both `Car` and `Motorcycle` — the two classes our parking lot needs.

Two versions are available as Kaggle inputs:

- **`vehicles-openimages`** — standard YOLO format (xywh labels), used by the default `yolov8` variant
- **`vehicles-openimages-obb`** — OBB format (4-corner rotated labels), used by the `yolo26_obb` variant

The active dataset is selected by `MODEL_VARIANT` in Section 3.

**Make sure you've added the zip as a Kaggle Input dataset (see the steps above) before running this.**

In [ ]:
# Dataset path (explicit — no glob searching)
DATASET_DIR = "/kaggle/input/datasets/tomasreisenman/vehicles-openimages"
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

print(f"Using dataset: {DATASET_DIR}")
print(f"Using model: {MODEL_NAME}")

## 5. Inspect the dataset structure

Before training, let's sanity-check what we actually downloaded: the `data.yaml` (class names + folder layout) and how many images/labels exist per split.

In [ ]:
# data_yaml_path is already set in Section 4
with open(data_yaml_path, "r") as f:
    print(f.read())

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    if os.path.isdir(img_dir):
        n_images = len(glob.glob(os.path.join(img_dir, "*")))
        print(f"{split}: {n_images} images")
    else:
        print(f"{split}: folder not found")

## 6. Visual sanity check of the training data

Let's plot a handful of training images with their ground-truth bounding boxes drawn, just to confirm the labels look correct before we spend CPU time training on them.

In [ ]:
def yolo_txt_to_boxes(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = map(float, parts)
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append((int(cls), x1, y1, x2, y2))
    return boxes


train_img_dir = os.path.join(DATASET_DIR, "train", "images")
train_lbl_dir = os.path.join(DATASET_DIR, "train", "labels")
sample_imgs = glob.glob(os.path.join(train_img_dir, "*"))[:4]

fig, axes = plt.subplots(1, len(sample_imgs), figsize=(20, 5))
for ax, img_path in zip(axes, sample_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label_path = os.path.join(train_lbl_dir, Path(img_path).stem + ".txt")
    for cls, x1, y1, x2, y2 in yolo_txt_to_boxes(label_path, w, h):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Train YOLO

Training choices, and why — tuned for the default GPU runtime (T4 x2); flip the boolean in Section 3 to switch to CPU:

- **Model: `yolov8n.pt`** (nano) — the smallest YOLOv8 variant. On CPU, larger variants (s/m/l/x) are noticeably slower per epoch and not worth it for a first pass. On GPU (like your T4 x2) you have headroom to try `yolov8s.pt` instead if you want more accuracy — change `MODEL_NAME` in Section 3.
- **Image size: 416** instead of the default 640 — smaller images mean roughly 2-3x faster CPU inference/training with a moderate accuracy trade-off. On GPU this constraint matters less, but it's left as-is to keep results comparable across runs.
- **Epochs: 20** — enough to see the loss curves converge without an excessive run time on CPU. With a GPU this will finish much faster, so feel free to raise it.
- **Batch size: auto on GPU, 8 on CPU** — `batch=-1` tells `ultralytics` to automatically pick the largest batch size that fits in GPU memory, which uses your T4s far more efficiently than a fixed small batch. On CPU there's no such benefit, so it stays fixed at 8 to keep memory use predictable.

On CPU, expect this to take a while (potentially 30+ minutes). On GPU it should be a few minutes. Ultralytics prints per-epoch progress so you can monitor it either way.

In [ ]:
model = YOLO(MODEL_NAME)  # starts from COCO-pretrained weights, then fine-tunes on our dataset

train_batch = -1 if FORCE_DEVICE != "cpu" else 8  # auto-batch on GPU, fixed small batch on CPU

results = model.train(
    data=data_yaml_path,
    epochs=20,
    imgsz=416,
    batch=train_batch,
    device=FORCE_DEVICE,   # from Section 3 — 0 (first GPU) by default, "cpu" if USE_CPU
    workers=2,
    # --- Data augmentation ---
    hsv_h=0.015,      # slight hue jitter
    hsv_s=0.7,        # saturation jitter — helps with lighting/paint-color variety
    hsv_v=0.4,        # brightness jitter — useful given your day/night frames
    degrees=15.0,     # moderate rotation
    translate=0.1,    # small positional jitter
    scale=0.5,        # scale jitter — helps with camera-distance/zoom mismatch
    shear=2.0,        # mild shear — approximates minor perspective variation
    perspective=0.0002,
    flipud=0.0,       # vertical flip — disabled: camera is oblique, not top-down
    fliplr=0.5,       # horizontal flip, standard
    mosaic=1.0,       # combine 4 images per batch
    mixup=0.1,        # light regularization
    erasing=0.4,      # random erasing
    project="parking_yolo",
    name="train_run",
    patience=10,           # stop early if validation stops improving
)

BEST_WEIGHTS = os.path.join(results.save_dir, "weights", "best.pt")
print(f"Best weights saved at: {BEST_WEIGHTS}")

## 8. Validate the trained model

Run the built-in validation to get standard detection metrics (precision, recall, mAP@0.5, mAP@0.5:0.95) on the held-out validation split. These numbers tell us how much to trust the detector before relying on it for occupancy/tracking logic.

In [ ]:
trained_model = YOLO(BEST_WEIGHTS)
metrics = trained_model.val(data=data_yaml_path, imgsz=416, device=FORCE_DEVICE)

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## 9. Quick inference sanity check

Before moving to parking-lot-specific logic, let's confirm the trained model draws sensible boxes on a few validation images.

In [ ]:
val_img_dir = os.path.join(DATASET_DIR, "valid", "images")
test_samples = glob.glob(os.path.join(val_img_dir, "*"))[:4]

pred_results = trained_model.predict(test_samples, imgsz=416, device=FORCE_DEVICE, conf=0.25)

fig, axes = plt.subplots(1, len(pred_results), figsize=(20, 5))
for ax, r in zip(axes, pred_results):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Auto-calibrate parking spots from a reference image

Instead of hard-coding polygon coordinates, this section uses a **reference image with cars already parked** and the trained model to automatically extract the parking spots.

**How it works:**
1. Load the reference image (cars parked in their spots)
2. Run the trained model to detect all vehicles
3. For each detected vehicle, create a parking spot polygon around it (bounding box expanded by `SPOT_EXPANSION` to approximate the real spot boundaries)
4. Visualize the extracted spots for verification

**One-time per camera position** — if the camera moves, re-run this section with a new reference image.

In [ ]:
# --- Auto-calibrate parking spots from a reference image with cars ---
CALIBRATION_IMAGE_PATH = "/kaggle/input/datasets/tomasreisenman/reference/reference/20260912171801.jpg"

# Expansion factor: cars are smaller than spots, so we expand each detection
# bounding box by this fraction on each side to approximate the spot boundaries.
# Increase if spots are too tight, decrease if they overlap too much.
SPOT_EXPANSION = 0.3

# Load calibration image and run detection
calib_img_bgr = cv2.imread(CALIBRATION_IMAGE_PATH)
if calib_img_bgr is None:
    raise FileNotFoundError(f"Could not read {CALIBRATION_IMAGE_PATH}")

calib_result = trained_model.predict(CALIBRATION_IMAGE_PATH, imgsz=416, device=FORCE_DEVICE, conf=0.25, verbose=False)[0]

# Extract parking spots from detections
PARKING_SPOTS = {}
spot_idx = 1
for b in calib_result.boxes:
    cls_name = calib_result.names[int(b.cls)]
    if cls_name not in ("Car", "Motorcycle"):
        continue
    x1, y1, x2, y2 = b.xyxy[0].tolist()
    w = x2 - x1
    h = y2 - y1
    # Expand bounding box to approximate spot size
    x1_exp = max(0, int(x1 - w * SPOT_EXPANSION))
    y1_exp = max(0, int(y1 - h * SPOT_EXPANSION))
    x2_exp = min(calib_img_bgr.shape[1], int(x2 + w * SPOT_EXPANSION))
    y2_exp = min(calib_img_bgr.shape[0], int(y2 + h * SPOT_EXPANSION))
    # Create polygon (rectangle in image coordinates)
    polygon = [(x1_exp, y1_exp), (x2_exp, y1_exp), (x2_exp, y2_exp), (x1_exp, y2_exp)]
    vehicle_type = "motorcycle" if cls_name == "Motorcycle" else "car"
    PARKING_SPOTS[f"spot_{spot_idx}_{vehicle_type}"] = {
        "polygon": polygon,
        "vehicle_type": vehicle_type
    }
    spot_idx += 1

print(f"Extracted {len(PARKING_SPOTS)} parking spots from calibration image:")
for name, spot in PARKING_SPOTS.items():
    print(f"  {name}: {spot['vehicle_type']}  polygon={spot['polygon']}")

# Visualize the extracted spots
def draw_calibration(image_path, spots):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    overlay = img.copy()
    for name, spot in spots.items():
        pts = np.array(spot["polygon"], dtype=np.int32)
        color = (0, 255, 255) if spot["vehicle_type"] == "car" else (255, 0, 255)
        cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=3)
        cx, cy = pts.mean(axis=0).astype(int)
        cv2.putText(overlay, name, (cx - 40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(10, 8))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title("Auto-extracted parking spots — do these align with the real spots?")
    plt.show()

draw_calibration(CALIBRATION_IMAGE_PATH, PARKING_SPOTS)

## 11. Per-spot occupancy detection

For each calibrated spot, we check whether any detected vehicle box of the **matching class** overlaps it enough to count as "occupied":

- `vehicle_type == "car"` → only `Car` detections are considered.
- `vehicle_type == "motorcycle"` → only `Motorcycle` detections are considered (as agreed, this spot never checks for cars, brand, or model).

Overlap is measured as **intersection area / spot polygon area** using `shapely` (not a plain box IoU, since spots are irregular quadrilaterals). A spot is "occupied" if this ratio passes a threshold (default 30% — tune this once you see it running on real photos).

In [ ]:
OCCUPANCY_THRESHOLD = 0.80

def check_occupancy(image_path, model, spots, conf=0.25, imgsz=416):
    result = model.predict(image_path, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    names = result.names  # class id -> class name

    detections = []
    for b in result.boxes:
        cls_name = names[int(b.cls)]
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        detections.append({"class": cls_name, "box": (x1, y1, x2, y2), "conf": float(b.conf)})

    occupancy = {}
    for spot_name, spot in spots.items():
        spot_poly = Polygon(spot["polygon"])
        expected_class = "Car" if spot["vehicle_type"] == "car" else "Motorcycle"

        occupied = False
        matched_box = None
        for det in detections:
            if det["class"] != expected_class:
                continue
            det_poly = shapely_box(*det["box"])
            if not spot_poly.intersects(det_poly):
                continue
            overlap_ratio = spot_poly.intersection(det_poly).area / spot_poly.area
            if overlap_ratio >= OCCUPANCY_THRESHOLD:
                occupied = True
                matched_box = det["box"]
                break

        occupancy[spot_name] = {"occupied": occupied, "box": matched_box}

    return occupancy, detections

## 12. Run occupancy check on a single frame and visualize it

Green outline = occupied, red outline = empty — a quick visual confirmation that matches what a human glancing at the photo would say.

The cell also verifies that `TEST_FRAME_PATH` and every image in `TEST_IMAGES` (used by the two debug cells below) actually exists under `BASE_DIR`.

In [ ]:
def visualize_occupancy(image_path, occupancy, spots):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    for spot_name, spot in spots.items():
        pts = np.array(spot["polygon"], dtype=np.int32)
        is_occupied = occupancy[spot_name]["occupied"]
        color = (0, 200, 0) if is_occupied else (220, 0, 0)
        cv2.polylines(img, [pts], isClosed=True, color=color, thickness=3)
        label = f"{spot_name}: {'OCCUPIED' if is_occupied else 'EMPTY'}"
        cx, cy = pts.mean(axis=0).astype(int)
        cv2.putText(img, label, (cx - 60, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

BASE_DIR = "/kaggle/input/datasets/tomasreisenman/reference/reference/"

TEST_FRAME_PATH = BASE_DIR + "20260915212238" + ".jpg"

TEST_IMAGES = [
    BASE_DIR + "20260910200600.jpg",
    BASE_DIR + "20260911172532.jpg",
    BASE_DIR + "20260911194005.jpg",
    BASE_DIR + "20260912171801.jpg",
    BASE_DIR + "20260912191134.jpg",
    BASE_DIR + "20260913005125.jpg",
    BASE_DIR + "20260914191018.jpg",
    BASE_DIR + "20260915212238.jpg",
    BASE_DIR + "20260916110642.jpg",
    BASE_DIR + "20260920190053.jpg",
    BASE_DIR + "reference_image.jpg",
]

missing = [p for p in [TEST_FRAME_PATH] + TEST_IMAGES if not os.path.exists(p)]
if missing:
    print("Warning — not found (check the dataset was added and BASE_DIR is correct):")
    for p in missing:
        print(f"  {p}")
else:
    print(f"All {len(TEST_IMAGES)} test images found.")

if os.path.exists(TEST_FRAME_PATH):
    occupancy, detections = check_occupancy(TEST_FRAME_PATH, trained_model, PARKING_SPOTS)
    print(occupancy)
    visualize_occupancy(TEST_FRAME_PATH, occupancy, PARKING_SPOTS)
else:
    print("TEST_FRAME_PATH not found — skipping the occupancy check.")

In [ ]:
# See every raw detection with its confidence, before spot-matching logic touches it
raw_results = {}
for img_path in TEST_IMAGES:
    if not os.path.exists(img_path):
        print(f"Skipping missing: {img_path}")
        continue
    result = trained_model.predict(img_path, imgsz=416, device=FORCE_DEVICE, conf=0.25, verbose=False)[0]
    raw_results[img_path] = result
    print(f"\n{os.path.basename(img_path)}")
    for b in result.boxes:
        box = [round(v, 1) for v in b.xyxy[0].tolist()]
        print(f"  {result.names[int(b.cls)]}  conf={float(b.conf):.3f}  box={box}")

In [ ]:
# Visualize the raw boxes directly on the image
import matplotlib.pyplot as plt

paths = [p for p in TEST_IMAGES if p in raw_results]
cols = 4
rows = (len(paths) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
for ax, img_path in zip(axes.flat, paths):
    ax.imshow(cv2.cvtColor(raw_results[img_path].plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis("off")
for ax in axes.flat[len(paths):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
import glob

counts = Counter()
for label_file in glob.glob(f"{DATASET_DIR}/train/labels/*.txt"):
    with open(label_file) as f:
        for line in f:
            cls_id = int(line.split()[0])
            counts[cls_id] += 1

print(counts)  # compare count for class id 3 (Motorcycle) vs class id 2 (Car)

## 12b. Homography experiment — blur + greyscale, then perspective warp

The idea: warp the camera's oblique view toward a bird's-eye view before running detection, on the theory that it may look more like overhead parking-lot imagery.

Important caveat: a homography is only geometrically exact for things lying flat on the ground plane it was computed from. Cars and motorcycles have height, so their upper parts won't warp correctly — this can help or hurt detection, which is exactly why we test it visually here rather than assuming it works.

**Preprocessing first:** line detection runs on a blurred greyscale image, so the Hough transform finds only long structural lines (spot edges, curbs) and doesn't pick up useless shapes from asphalt grain, shadows, or leaves.

**Then we try combinations:** detection on the original vs. the warped image (and warped+blur), across all `TEST_IMAGES`, to see if the warp actually helps.

In [ ]:
# --- Step 1: preprocess (blur + greyscale) before line detection ---
# Blur + greyscale removes asphalt grain, shadows and other fine texture so the
# Hough transform only finds long structural lines (spot edges, curbs), not
# useless small shapes.

def preprocess_for_lines(image_bgr, blur_kernel=(5, 5)):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.GaussianBlur(gray, blur_kernel, 0)


def filter_border_lines(lines, image_shape, border_threshold=10, angle_threshold=10):
    """Discard lines that are nearly parallel to an image border AND close to it.
    
    The Hough transform often detects the image border itself as a line,
    which is not a useful structural line for homography computation.
    """
    if len(lines) == 0:
        return lines
    h, w = image_shape[:2]
    filtered = []
    for line in lines:
        x1, y1, x2, y2 = line
        angle = line_angle_deg(x1, y1, x2, y2)
        # Check if nearly horizontal (0° or 180°) and close to top/bottom border
        is_horizontal = angle < angle_threshold or angle > (180 - angle_threshold)
        if is_horizontal:
            y_min = min(y1, y2)
            y_max = max(y1, y2)
            if y_min < border_threshold or y_max > (h - border_threshold):
                continue  # skip this line
        # Check if nearly vertical (90°) and close to left/right border
        is_vertical = (90 - angle_threshold) < angle < (90 + angle_threshold)
        if is_vertical:
            x_min = min(x1, x2)
            x_max = max(x1, x2)
            if x_min < border_threshold or x_max > (w - border_threshold):
                continue  # skip this line
        filtered.append(line)
    return np.array(filtered, dtype=int) if filtered else np.empty((0, 4), dtype=int)


def detect_line_segments(image_bgr, canny_low=50, canny_high=150,
                          hough_threshold=60, min_line_length=60, max_line_gap=15):
    gray = preprocess_for_lines(image_bgr)
    edges = cv2.Canny(gray, canny_low, canny_high)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=hough_threshold,
                             minLineLength=min_line_length, maxLineGap=max_line_gap)
    if lines is None:
        return np.empty((0, 4), dtype=int)
    lines = lines.reshape(-1, 4)
    # Filter out lines that are too close to the image borders
    lines = filter_border_lines(lines, image_bgr.shape, border_threshold=10, angle_threshold=10)
    return lines


def line_angle_deg(x1, y1, x2, y2):
    return np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180  # direction only, 0-180


def angular_dist(a, b):
    d = abs(a - b) % 180
    return min(d, 180 - d)


def cluster_lines_by_angle(lines, min_group_size=4, min_separation_deg=20, bin_size=2):
    if len(lines) == 0:
        return None, None
    angles = np.array([line_angle_deg(*l) for l in lines])
    bins = np.arange(0, 180 + bin_size, bin_size)
    hist, edges_ = np.histogram(angles, bins=bins)

    peak1_idx = np.argmax(hist)
    peak1_angle = (edges_[peak1_idx] + edges_[peak1_idx + 1]) / 2

    hist_suppressed = hist.copy()
    for i, (lo, hi) in enumerate(zip(edges_[:-1], edges_[1:])):
        if angular_dist((lo + hi) / 2, peak1_angle) < min_separation_deg:
            hist_suppressed[i] = 0

    if hist_suppressed.max() == 0:
        return None, None  # no second, sufficiently distinct direction found

    peak2_idx = np.argmax(hist_suppressed)
    peak2_angle = (edges_[peak2_idx] + edges_[peak2_idx + 1]) / 2

    g1_mask = np.array([angular_dist(a, peak1_angle) < angular_dist(a, peak2_angle) for a in angles])
    group1, group2 = lines[g1_mask], lines[~g1_mask]

    if len(group1) < min_group_size or len(group2) < min_group_size:
        return None, None
    return group1, group2


def estimate_vanishing_point(lines):
    def to_homog_line(x1, y1, x2, y2):
        return np.cross([x1, y1, 1.0], [x2, y2, 1.0])

    L = np.array([to_homog_line(*l) for l in lines])
    _, _, VT = np.linalg.svd(L)
    p = VT[-1]
    return p / p[-1] if abs(p[-1]) > 1e-8 else p


def line_length(line):
    x1, y1, x2, y2 = line
    return np.hypot(x2 - x1, y2 - y1)


def longest_line(lines):
    """Return the single longest line from the array."""
    if len(lines) == 0:
        return None
    lengths = np.array([line_length(l) for l in lines])
    return lines[np.argmax(lengths)]


def compute_auto_homography(image_bgr, **hough_kwargs):
    lines = detect_line_segments(image_bgr, **hough_kwargs)
    if len(lines) < 8:
        return None, lines, None

    group1, group2 = cluster_lines_by_angle(lines)
    if group1 is None:
        return None, lines, None

    # Use only the longest line from each group
    longest1 = longest_line(group1)
    longest2 = longest_line(group2)

    v1 = estimate_vanishing_point(np.array([longest1]))
    v2 = estimate_vanishing_point(np.array([longest2]))
    vanishing_line = np.cross(v1, v2)
    if abs(vanishing_line[2]) < 1e-8:
        return None, lines, (group1, group2)

    l1, l2, _ = vanishing_line / vanishing_line[2]
    H = np.array([[1, 0, 0], [0, 1, 0], [l1, l2, 1]], dtype=np.float64)
    return H, lines, (group1, group2)


def warp_with_auto_canvas(image_bgr, H, max_scale=4.0):
    h, w = image_bgr.shape[:2]
    corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float64)
    corners_h = np.hstack([corners, np.ones((4, 1))])
    warped_corners_h = (H @ corners_h.T).T
    warped_corners = warped_corners_h[:, :2] / warped_corners_h[:, 2:3]

    min_xy, max_xy = warped_corners.min(axis=0), warped_corners.max(axis=0)
    size = max_xy - min_xy

    if not np.all(np.isfinite(size)) or size.max() > max_scale * max(w, h) or size.min() <= 1:
        return None, None  # degenerate homography — canvas would explode or collapse

    translation = np.array([[1, 0, -min_xy[0]], [0, 1, -min_xy[1]], [0, 0, 1]])
    H_full = translation @ H
    warped = cv2.warpPerspective(image_bgr, H_full, (int(size[0]), int(size[1])))
    return warped, H_full


# --- Run the auto-calibration once, on the clearest reference photo ---
REFERENCE_IMAGE_FOR_HOMOGRAPHY = BASE_DIR + "reference_image.jpg"
ref_img = cv2.imread(REFERENCE_IMAGE_FOR_HOMOGRAPHY)
if ref_img is None:
    raise FileNotFoundError(f"Could not read {REFERENCE_IMAGE_FOR_HOMOGRAPHY}")

H_raw, detected_lines, groups = compute_auto_homography(ref_img)

# Show what the preprocessing + line detection actually sees, for debugging.
preprocessed = preprocess_for_lines(ref_img)
edges_vis = cv2.Canny(preprocessed, 50, 150)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Reference (original)")
axes[0].axis("off")
axes[1].imshow(preprocessed, cmap="gray")
axes[1].set_title("Blurred greyscale (line-detection input)")
axes[1].axis("off")
axes[2].imshow(edges_vis, cmap="gray")
axes[2].set_title("Canny edges")
axes[2].axis("off")
plt.tight_layout()
plt.show()

# Visualize what was detected, success or not, so failures are debuggable.
lines_vis = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB).copy()
if groups is not None:
    for x1, y1, x2, y2 in groups[0]:
        cv2.line(lines_vis, (x1, y1), (x2, y2), (255, 0, 0), 2)     # direction 1: red
    for x1, y1, x2, y2 in groups[1]:
        cv2.line(lines_vis, (x1, y1), (x2, y2), (0, 200, 0), 2)     # direction 2: green
    title = f"{len(detected_lines)} lines detected — grouped into two directions (red / green)"
else:
    for x1, y1, x2, y2 in detected_lines:
        cv2.line(lines_vis, (x1, y1), (x2, y2), (255, 255, 0), 1)
    title = f"{len(detected_lines)} lines detected — could not find two distinct directions"

plt.figure(figsize=(9, 7))
plt.imshow(lines_vis)
plt.title(title)
plt.axis("off")
plt.show()

HOMOGRAPHY_MATRIX, OUTPUT_SIZE = None, None

if H_raw is not None:
    warped, H_final = warp_with_auto_canvas(ref_img, H_raw)
    if warped is not None:
        HOMOGRAPHY_MATRIX, OUTPUT_SIZE = H_final, (warped.shape[1], warped.shape[0])
        plt.figure(figsize=(9, 7))
        plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
        plt.title("Auto-rectified (perspective skew removed)")
        plt.axis("off")
        plt.show()
    else:
        print("Homography looked degenerate (canvas would explode/collapse) — skipping warp.")
else:
    print("Could not reliably detect two distinct line directions in the reference image.")
    print("Try adjusting detect_line_segments()'s canny_low/canny_high/hough_threshold above, "
          "or pick a different, clearer reference frame.")

In [ ]:
def apply_homography(image_bgr):
    if HOMOGRAPHY_MATRIX is None:
        return image_bgr  # no valid transform found — pass through unchanged
    return cv2.warpPerspective(image_bgr, HOMOGRAPHY_MATRIX, OUTPUT_SIZE)


def run_and_show(image_path, model, imgsz=416, conf=0.25):
    print(f"\n=== {os.path.basename(image_path)} ===")

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print("  Could not read image, skipping.")
        return

    warped_bgr = apply_homography(img_bgr)

    orig_result = model.predict(img_bgr, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    warped_result = model.predict(warped_bgr, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]

    print("  Original detections:")
    for b in orig_result.boxes:
        print(f"    {orig_result.names[int(b.cls)]}  conf={float(b.conf):.2f}  box={b.xyxy[0].tolist()}")
    print("  Warped detections:")
    for b in warped_result.boxes:
        print(f"    {warped_result.names[int(b.cls)]}  conf={float(b.conf):.2f}  box={b.xyxy[0].tolist()}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(orig_result.plot(), cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(warped_result.plot(), cv2.COLOR_BGR2RGB))
    axes[1].set_title("Warped")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


run_and_show(TEST_FRAME_PATH, trained_model)

In [ ]:
# --- Try combinations across all test images: original / warped / warped+blur ---
combo_rows = []
for img_path in TEST_IMAGES:
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue
    warped_bgr = apply_homography(img_bgr)
    variants = {
        "original": img_bgr,
        "warped": warped_bgr,
        "warped+blur": cv2.GaussianBlur(warped_bgr, (5, 5), 0),
    }
    for name, variant in variants.items():
        res = trained_model.predict(variant, imgsz=416, device=FORCE_DEVICE, conf=0.25, verbose=False)[0]
        confs = [float(b.conf) for b in res.boxes]
        combo_rows.append({
            "image": os.path.basename(img_path),
            "variant": name,
            "n_detections": len(confs),
            "mean_conf": round(float(np.mean(confs)), 3) if confs else 0.0,
        })

combo_df = pd.DataFrame(combo_rows)
pivot = combo_df.pivot_table(index="image", columns="variant", values="n_detections", aggfunc="first")
print(pivot)

## 12c. Parking Lot Alignment Strategy — black & yellow wheel stoppers

An alternative to the line-based homography (Section 12b): instead of detecting lines, this strategy uses the **black and yellow wheel stoppers** as alignment references. The four steps:

1. **HSV Color Filtering** — Convert the image to HSV and mask out the bright yellow stripes on the concrete wheel stops, isolating them from the black asphalt and surrounding cars.
2. **Morphological Closing** — Apply a large structural kernel to "bridge" the black stripes, fusing fragmented yellow sections into solid rectangular blocks.
3. **Contour & Angle Extraction** — Find block shapes using `cv2.findContours` and calculate their exact tilt using `cv2.minAreaRect`. Compute the median angle across all detected blocks.
4. **Warp Transformation** — Rotate the entire image via `cv2.warpAffine` using the inverse median angle to align the parking spots perpendicular to the horizontal axis.

In [ ]:
# --- Step 1: HSV color filtering — isolate the yellow stripes ---
def filter_yellow_stops(image_bgr):
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    lower_yellow = np.array([15, 80, 80])
    upper_yellow = np.array([40, 255, 255])
    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)
    return mask


# --- Step 2: Morphological closing — fuse fragmented yellow sections ---
def solidify_stops(mask, kernel_size=31):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return closed


# --- Step 3: Contour & angle extraction ---
def extract_stop_angles(closed_mask, min_area=800):
    contours, _ = cv2.findContours(closed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    angles = []
    valid_contours = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        rect = cv2.minAreaRect(cnt)
        angle = rect[2]
        # Normalize to [0, 90) for easier interpretation
        if angle < -45:
            angle += 90
        angles.append(angle)
        valid_contours.append(cnt)
    return angles, valid_contours


# --- Step 4: Warp transformation ---
def align_parking(image_bgr, median_angle):
    h, w = image_bgr.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, -median_angle, 1.0)
    aligned = cv2.warpAffine(image_bgr, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return aligned


# --- Run the pipeline on the reference image ---
ref_bgr = cv2.imread(REFERENCE_IMAGE_FOR_HOMOGRAPHY)
if ref_bgr is None:
    raise FileNotFoundError(f"Could not read {REFERENCE_IMAGE_FOR_HOMOGRAPHY}")

yellow_mask = filter_yellow_stops(ref_bgr)
closed_mask = solidify_stops(yellow_mask, kernel_size=31)
angles, valid_contours = extract_stop_angles(closed_mask, min_area=800)

if len(angles) == 0:
    print("No wheel stoppers detected. Try adjusting the HSV thresholds or kernel size.")
    MEDIAN_ANGLE = 0.0
else:
    MEDIAN_ANGLE = float(np.median(angles))
    print(f"Detected {len(angles)} wheel stopper blocks")
    print(f"Angles: {[round(a, 2) for a in angles]}")
    print(f"Median angle: {MEDIAN_ANGLE:.2f} degrees")

# Visualize each step
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
axes[0].imshow(cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(yellow_mask, cmap="gray")
axes[1].set_title("HSV Yellow Mask")
axes[1].axis("off")
axes[2].imshow(closed_mask, cmap="gray")
axes[2].set_title("After Morphological Closing")
axes[2].axis("off")
contour_vis = cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB).copy()
cv2.drawContours(contour_vis, valid_contours, -1, (255, 0, 0), 2)
axes[3].imshow(contour_vis)
axes[3].set_title(f"Detected Blocks (median: {MEDIAN_ANGLE:.1f}°)")
axes[3].axis("off")
plt.tight_layout()
plt.show()

# Step 4: Warp and visualize
aligned_bgr = align_parking(ref_bgr, MEDIAN_ANGLE)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Aligned (rotated {-MEDIAN_ANGLE:.1f}°)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Compare detection: original vs wheel-stopper-aligned ---
def run_alignment_comparison(image_path, model, imgsz=416, conf=0.25):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f"Could not read {image_path}")
        return
    aligned = align_parking(img_bgr, MEDIAN_ANGLE)
    orig_result = model.predict(img_bgr, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    aligned_result = model.predict(aligned, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    print(f"\n=== {os.path.basename(image_path)} ===")
    print(f"  Original: {len(orig_result.boxes)} detections")
    print(f"  Aligned:  {len(aligned_result.boxes)} detections")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(orig_result.plot(), cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(aligned_result.plot(), cv2.COLOR_BGR2RGB))
    axes[1].set_title("Aligned")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


run_alignment_comparison(TEST_FRAME_PATH, trained_model)

In [ ]:
# --- Try combinations: original vs aligned across all test images ---
align_rows = []
for img_path in TEST_IMAGES:
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue
    aligned = align_parking(img_bgr, MEDIAN_ANGLE)
    for name, variant in {"original": img_bgr, "aligned": aligned}.items():
        res = trained_model.predict(variant, imgsz=416, device=FORCE_DEVICE, conf=0.25, verbose=False)[0]
        confs = [float(b.conf) for b in res.boxes]
        align_rows.append({
            "image": os.path.basename(img_path),
            "variant": name,
            "n_detections": len(confs),
            "mean_conf": round(float(np.mean(confs)), 3) if confs else 0.0,
        })

align_df = pd.DataFrame(align_rows)
print(align_df.pivot_table(index="image", columns="variant", values="n_detections", aggfunc="first"))

## 13. Whole-lot tracking: entries, exits, and duration

Since the camera captures **still photos every N minutes** (not continuous video), a heavyweight tracker like SORT is overkill — its Kalman-filter motion model assumes smooth frame-to-frame movement, which doesn't hold when minutes pass between photos. Instead we use simple, explainable **centroid tracking**:

- For each new frame, detect all `Car` boxes and compute their centroids.
- Match each centroid to the closest centroid from the previous frame (within a distance threshold) → same car, same ID.
- Unmatched **new** centroids → a car entered.
- Previously tracked centroids with **no match** in the new frame → that car left.
- Duration = time between a car's first-seen and last-seen frame timestamp.

**On timestamps:** as agreed, we do **not** read the timestamp burned into the image (no OCR, nothing that touches the training or detection pipeline). Instead, timestamps are supplied externally — e.g. parsed from filenames like `frame_2026-09-13_00-51.jpg`, or from a separate log you already keep when saving the photos. The tracker just needs a `(image_path, timestamp)` pair per frame.

In [ ]:
from datetime import datetime

class CentroidTracker:
    def __init__(self, max_distance=80):
        self.max_distance = max_distance
        self.next_id = 0
        self.active = {}   # id -> {"centroid": (x, y), "first_seen": ts, "last_seen": ts}
        self.log = []      # finished entries: {"id", "entered", "left", "duration"}

    @staticmethod
    def _centroid(box):
        x1, y1, x2, y2 = box
        return ((x1 + x2) / 2, (y1 + y2) / 2)

    def update(self, boxes, timestamp):
        new_centroids = [self._centroid(b) for b in boxes]
        matched_ids = set()
        used_new = set()

        # try to match existing tracked cars to the closest new centroid
        for obj_id, obj in list(self.active.items()):
            best_idx, best_dist = None, None
            for idx, c in enumerate(new_centroids):
                if idx in used_new:
                    continue
                dist = np.hypot(c[0] - obj["centroid"][0], c[1] - obj["centroid"][1])
                if best_dist is None or dist < best_dist:
                    best_idx, best_dist = idx, dist
            if best_idx is not None and best_dist <= self.max_distance:
                self.active[obj_id]["centroid"] = new_centroids[best_idx]
                self.active[obj_id]["last_seen"] = timestamp
                used_new.add(best_idx)
                matched_ids.add(obj_id)

        # anything not matched this round has left -> close it out
        for obj_id in list(self.active.keys()):
            if obj_id not in matched_ids:
                obj = self.active.pop(obj_id)
                self.log.append({
                    "id": obj_id,
                    "entered": obj["first_seen"],
                    "left": obj["last_seen"],
                    "duration": obj["last_seen"] - obj["first_seen"],
                })

        # any leftover new centroid is a brand-new car entering
        for idx, c in enumerate(new_centroids):
            if idx not in used_new:
                self.active[self.next_id] = {"centroid": c, "first_seen": timestamp, "last_seen": timestamp}
                self.next_id += 1

    def finalize(self, final_timestamp):
        # call this after the last frame, so cars still parked show up in the log too
        for obj_id, obj in self.active.items():
            self.log.append({
                "id": obj_id,
                "entered": obj["first_seen"],
                "left": None,  # still there as of the last frame
                "duration": final_timestamp - obj["first_seen"],
            })

## 14. Process a sequence of frames

`FRAME_SEQUENCE` is a list of `(image_path, timestamp)` pairs, ordered by time. Replace the example below with your real periodic photos and their real timestamps (e.g. parsed from filenames or your capture log — not from OCR on the image).

For each frame we run detection, keep only `Car` boxes (motorcycles are handled separately by the occupancy check above, not by whole-lot entry/exit tracking), and feed them to the tracker.

In [ ]:
# Example structure — replace with your real captured frames and timestamps.
FRAME_SEQUENCE = [
    # ("frames/frame_2026-09-13_00-40.jpg", datetime(2026, 9, 13, 0, 40)),
    # ("frames/frame_2026-09-13_00-50.jpg", datetime(2026, 9, 13, 0, 50)),
    # ("frames/frame_2026-09-13_01-00.jpg", datetime(2026, 9, 13, 1, 0)),
]

tracker = CentroidTracker(max_distance=80)

for image_path, timestamp in FRAME_SEQUENCE:
    result = trained_model.predict(image_path, imgsz=416, device=FORCE_DEVICE, conf=0.25, verbose=False)[0]
    names = result.names
    car_boxes = [
        b.xyxy[0].tolist()
        for b in result.boxes
        if names[int(b.cls)] == "Car"
    ]
    tracker.update(car_boxes, timestamp)

if FRAME_SEQUENCE:
    tracker.finalize(FRAME_SEQUENCE[-1][1])
    print(f"Processed {len(FRAME_SEQUENCE)} frames.")
else:
    print("Add real (image_path, timestamp) pairs to FRAME_SEQUENCE to run this.")

## 15. Entry / exit / duration log

The tracker's log, as a table: one row per vehicle, with when it entered, when it left (or `None` if still parked as of the last frame), and total duration.

In [ ]:
log_df = pd.DataFrame(tracker.log)
if not log_df.empty:
    log_df = log_df.sort_values("entered").reset_index(drop=True)
log_df

## 16. Final module: single-photo vehicle report (color, brand, model)

This last part is a **separate, standalone step**: given one parking-lot photo, detect each vehicle and report:
- **Color** (both cars and motorcycles) — computed directly from pixels, no extra model needed.
- **Brand and model** (**cars only**, per what we agreed — motorcycles skip this) — using **CLIP zero-shot classification**: we give CLIP the cropped vehicle image plus a list of candidate brand/model text labels, and it scores how well each label matches, with no additional training required.

First, load the pretrained CLIP model (this only downloads/loads weights — it is not trained or fine-tuned on our data at all).

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

print("CLIP model loaded (CPU, inference-only).")

## 17. Dominant color extraction

For a cropped vehicle image, we cluster its pixels into a few groups with K-Means and pick the largest cluster's average color, then map that RGB value to the closest common car-color name (a lightweight lookup, not a trained model).

In [ ]:
from sklearn.cluster import KMeans

COMMON_COLORS = {
    "white": (255, 255, 255), "black": (0, 0, 0), "gray": (128, 128, 128),
    "silver": (192, 192, 192), "red": (200, 30, 30), "blue": (30, 60, 180),
    "green": (30, 120, 60), "yellow": (230, 210, 40), "orange": (230, 120, 30),
    "brown": (110, 70, 40),
}

def dominant_color_name(crop_rgb, n_clusters=3):
    pixels = crop_rgb.reshape(-1, 3)
    # skip near-black/near-white shadow or glare pixels? keep it simple for now, use all pixels.
    kmeans = KMeans(n_clusters=n_clusters, n_init=4, random_state=0).fit(pixels)
    counts = np.bincount(kmeans.labels_)
    dominant_rgb = kmeans.cluster_centers_[np.argmax(counts)]

    best_name, best_dist = None, None
    for name, ref_rgb in COMMON_COLORS.items():
        dist = np.linalg.norm(dominant_rgb - np.array(ref_rgb))
        if best_dist is None or dist < best_dist:
            best_name, best_dist = name, dist
    return best_name

## 18. Brand/model zero-shot classification + combined report

`CAR_BRAND_MODEL_CANDIDATES` is an editable list of "Brand Model" text labels CLIP will choose from — it can only pick among what you give it, so expand this list with whatever brands/models are realistic for your parking lot. Accuracy on exact model/year is limited (this is a zero-shot general-purpose model, not a fine-tuned car classifier) — treat the output as a best guess, not ground truth.

In [ ]:
CAR_BRAND_MODEL_CANDIDATES = [
    "Toyota Corolla", "Toyota Hilux", "Ford Ka", "Ford Focus", "Chevrolet Onix",
    "Volkswagen Gol", "Volkswagen Golf", "Renault Sandero", "Renault Duster",
    "Fiat Cronos", "Peugeot 208", "Honda Civic", "Honda Fit", "Nissan Versa",
    "Hyundai HB20", "Kia Rio", "BMW 3 Series", "Mercedes-Benz C-Class", "Audi A3",
]

@torch.no_grad()
def guess_brand_model(crop_pil, candidates):
    image_input = clip_preprocess(crop_pil).unsqueeze(0)
    text_input = clip_tokenizer([f"a photo of a {c}" for c in candidates])

    image_features = clip_model.encode_image(image_input)
    text_features = clip_model.encode_text(text_input)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    similarity = (image_features @ text_features.T).squeeze(0)
    best_idx = similarity.argmax().item()
    return candidates[best_idx], float(similarity[best_idx])


def analyze_parking_photo(image_path, model, candidates=CAR_BRAND_MODEL_CANDIDATES, conf=0.25, imgsz=416):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    result = model.predict(image_path, imgsz=imgsz, device=FORCE_DEVICE, conf=conf, verbose=False)[0]
    names = result.names

    report = []
    for b in result.boxes:
        cls_name = names[int(b.cls)]
        if cls_name not in ("Car", "Motorcycle"):
            continue

        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        crop_rgb = img_rgb[y1:y2, x1:x2]
        if crop_rgb.size == 0:
            continue

        entry = {
            "type": cls_name.lower(),
            "box": (x1, y1, x2, y2),
            "confidence": float(b.conf),
            "color": dominant_color_name(crop_rgb),
        }

        if cls_name == "Car":
            crop_pil = Image.fromarray(crop_rgb)
            brand_model, score = guess_brand_model(crop_pil, candidates)
            entry["brand_model"] = brand_model
            entry["brand_model_confidence"] = round(score, 3)
        # Motorcycles intentionally get no brand/model field, per our agreed scope.

        report.append(entry)

    return report

## 19. Try it on a photo

Point `DEMO_IMAGE_PATH` at any parking-lot photo and run the full report.

In [ ]:
DEMO_IMAGE_PATH = BASE_DIR + "20260914191018.jpg"  # <-- replace with your own photo

if os.path.exists(DEMO_IMAGE_PATH):
    report = analyze_parking_photo(DEMO_IMAGE_PATH, trained_model)
    print(json.dumps(report, indent=2))
else:
    print("Upload a photo and set DEMO_IMAGE_PATH to run the final report.")

## 20. Notes and next steps

- **Fine-tuning on your own camera**: once you have real photos from your parking lot, labeling even 100-200 of them (e.g. with Roboflow's free annotation tool or `labelImg`) and continuing training from `best.pt` will likely improve accuracy more than anything else in this notebook — the current dataset is generic street/traffic imagery, not overhead parking shots.
- **Device handling**: one boolean (`USE_CPU`, Section 3) switches the whole notebook between the default GPU (device `0`) and CPU — nothing else needs to change.
- **Day/night robustness**: since your reference photo was a night shot, it's worth testing detection separately on day and night samples before trusting the counts — if night performance is notably worse, adding some night images to a fine-tuning round would help.
- **Timestamps**: this notebook deliberately never reads the burned-in camera timestamp. If you later want the tracker to source timestamps automatically instead of from filenames, that would be an OCR step (e.g. Tesseract) added as a fully separate utility — it should still never touch the training/detection pipeline itself.
- **Occupancy threshold**: `OCCUPANCY_THRESHOLD = 0.30` is a starting point — tune it against real photos of your spots partially blocked by shadows or nearby parked vehicles.